# Katube Colab - Sistema de Download de Audio do YouTube

Sistema modular para download de audio do YouTube com persistencia no Google Drive e gerenciamento completo de metadados.

**Recursos:**
- Download em 6 formatos de audio (mp3, flac, wav, m4a, opus, ogg)
- Extracao completa de metadados (18+ campos)
- CSV consolidado com historico de downloads
- Skip inteligente de duplicados
- Suporte a videos, playlists, canais e arquivos txt

---

## SECAO 1: Setup e Instalacao

Nesta secao:
1. Clonar o repositorio do GitHub
2. Instalar dependencias
3. Montar Google Drive
4. Criar estrutura de pastas

### 1.1 - Clonar Repositorio

Clona a branch com as implementacoes de metadados completos.

In [ ]:
# Remove pasta antiga se existir
!rm -rf katube-colab

# Clona repositorio na branch com metadados
!git clone --branch claude/katube-metadata-improvements-011CV6EzzLe4neGzos3kEBFE https://github.com/DosAnjos-AI/katube-colab.git

# Navega para o diretorio
%cd katube-colab

print("="*60)
print("Repositorio clonado com sucesso!")
print("="*60)

### 1.2 - Instalar Dependencias

Instala yt-dlp e pandas.

In [ ]:
# Instala dependencias do requirements.txt
!pip install -q -r requirements.txt

print("="*60)
print("Dependencias instaladas:")
print("  - yt-dlp (download de videos)")
print("  - pandas (gerenciamento de CSV)")
print("="*60)

### 1.3 - Montar Google Drive

Autentica e monta o Google Drive para persistencia de arquivos.

In [ ]:
from google.colab import drive

# Monta o Google Drive
drive.mount('/content/drive')

print("="*60)
print("Google Drive montado com sucesso!")
print("Localizacao: /content/drive/MyDrive")
print("="*60)

### 1.4 - Criar Estrutura de Pastas

Cria a estrutura de pastas no Drive para organizar downloads e logs.

In [ ]:
from downloaders import DriveManager
from config import Config

# Inicializa gerenciador do Drive
drive_manager = DriveManager()

# Cria estrutura de pastas
drive_manager.setup_folder_structure()

# Verifica espaco disponivel
space_info = drive_manager.check_drive_space()

if space_info['available']:
    print("\n" + "="*60)
    print("INFORMACOES DO GOOGLE DRIVE")
    print("="*60)
    print(f"Espaco total:      {space_info['total_formatted']}")
    print(f"Espaco usado:      {space_info['used_formatted']}")
    print(f"Espaco livre:      {space_info['free_formatted']}")
    print(f"Uso:               {space_info['usage_percent']:.1f}%")
    print("="*60)
else:
    print(f"AVISO: Nao foi possivel verificar espaco: {space_info.get('error')}")

## SECAO 2: Configuracoes

Configure todos os parametros de download nesta secao.

**IMPORTANTE:** Execute esta celula ANTES de iniciar os downloads.

In [ ]:
from config import Config

# ============================================================
# CONFIGURACOES PRINCIPAIS
# ============================================================

# URL do conteudo (video, playlist ou canal)
# Exemplos:
#   - Video:    "https://www.youtube.com/watch?v=VIDEO_ID"
#   - Playlist: "https://www.youtube.com/playlist?list=PLAYLIST_ID"
#   - Canal:    "https://www.youtube.com/@username"
URL = "COLE_A_URL_AQUI"

# ============================================================
# FORMATO DE AUDIO
# ============================================================

# Formato do arquivo de audio
# Opcoes disponiveis:
#   - 'mp3':   Formato universal, compativel com tudo (recomendado para musica)
#   - 'flac':  Audio sem perdas, melhor qualidade (recomendado para audiophiles)
#   - 'wav':   Audio sem perdas, sem compressao (arquivos muito grandes)
#   - 'm4a':   Boa qualidade, compressao eficiente (Apple/iTunes)
#   - 'opus':  Melhor compressao moderna (recomendado para podcasts)
#   - 'ogg':   Alternativa open-source ao MP3
AUDIO_FORMAT = 'mp3'

# ============================================================
# QUALIDADE DO AUDIO (kbps)
# ============================================================

# Qualidade do audio em kbps (kilobits por segundo)
# Opcoes:
#   0 ou 'best':  Maxima qualidade disponivel (RECOMENDADO para FLAC/WAV)
#                 - Extrai o melhor audio possivel do YouTube
#                 - Tamanho: varia (geralmente ~128-256 kbps no YouTube)
#
#   320:          Qualidade maxima para MP3
#                 - Som cristalino, indistinguivel do original
#                 - Tamanho: ~40 MB por hora de audio
#                 - Recomendado para: musica de alta qualidade, colecoes
#
#   256:          Alta qualidade (PADRAO RECOMENDADO)
#                 - Excelente qualidade, equilibrio ideal
#                 - Tamanho: ~32 MB por hora de audio
#                 - Recomendado para: uso geral, maioria dos casos
#
#   192:          Boa qualidade
#                 - Qualidade muito boa, tamanho moderado
#                 - Tamanho: ~24 MB por hora de audio
#                 - Recomendado para: musica casual, economizar espaco
#
#   128:          Qualidade basica
#                 - Qualidade aceitavel, arquivo pequeno
#                 - Tamanho: ~16 MB por hora de audio
#                 - Recomendado para: podcasts, audios falados, economizar muito espaco
#
# IMPORTANTE:
#   - Para FLAC/WAV: sempre use 0 (melhor qualidade)
#   - Para MP3: 256 ou 320 kbps sao ideais
#   - Para OPUS: 128 kbps ja oferece excelente qualidade
#   - Qualidade maior = arquivo maior
AUDIO_QUALITY = 256

# ============================================================
# FILTROS DE DURACAO
# ============================================================

# Duracao minima do video (em segundos)
# Exemplo: 30 = pula videos menores que 30 segundos
MIN_DURATION = 30

# Duracao maxima do video (em segundos)
# Exemplo: 7200 = pula videos maiores que 2 horas
#          0 = sem limite
MAX_DURATION = 7200  # 2 horas

# ============================================================
# CONTROLE DE DOWNLOADS
# ============================================================

# Pular videos ja baixados (recomendado: True)
# O sistema verifica:
#   1. Se o arquivo ja existe
#   2. Se o ID esta no CSV de metadados
#   3. Se a pasta existe
SKIP_EXISTING = True

# Delay entre downloads (em segundos)
# Ajuda a evitar bloqueios do YouTube
DELAY_MIN = 10  # Minimo
DELAY_MAX = 20  # Maximo

# ============================================================
# APLICAR CONFIGURACOES
# ============================================================

# Aplica as configuracoes ao Config global
Config.AUDIO_FORMAT = AUDIO_FORMAT
Config.AUDIO_QUALITY = AUDIO_QUALITY
Config.MIN_DURATION = MIN_DURATION
Config.MAX_DURATION = MAX_DURATION
Config.SKIP_EXISTING = SKIP_EXISTING
Config.DELAY_MIN = DELAY_MIN
Config.DELAY_MAX = DELAY_MAX

# Valida configuracoes
validation = Config.validate()

if validation['valid']:
    print("="*60)
    print("CONFIGURACOES APLICADAS COM SUCESSO")
    print("="*60)
    Config.print_config()
else:
    print("ERRO: Configuracoes invalidas!")
    for issue in validation['issues']:
        print(f"  - {issue}")

## SECAO 3: Download de Conteudo

Nesta secao:
1. Inicializar downloader e metadata manager
2. Processar URL
3. Salvar metadados automaticamente no CSV
4. Exibir estatisticas

### 3.1 - Inicializar Downloader

Cria instancias do YouTubeDownloader e MetadataManager.

In [ ]:
from downloaders import YouTubeDownloader, MetadataManager

# Inicializa o downloader com as configuracoes atuais
downloader = YouTubeDownloader()

# Inicializa o gerenciador de metadados
metadata_manager = MetadataManager()

print("="*60)
print("Downloader e MetadataManager inicializados")
print("="*60)
print(f"URL a processar: {URL}")
print(f"Formato de audio: {Config.AUDIO_FORMAT}")
print(f"Qualidade: {Config.AUDIO_QUALITY if Config.AUDIO_QUALITY > 0 else 'Melhor disponivel'}")
print(f"CSV de metadados: {Config.get_metadata_csv_path()}")
print("="*60)

### 3.2 - Executar Download

Processa a URL e faz o download automatico.

**O sistema automaticamente:**
- Detecta o tipo (video/playlist/canal)
- Verifica duplicados (arquivo + CSV + pasta)
- Faz o download do audio
- Extrai metadados completos (18+ campos)
- Salva no CSV consolidado
- Aplica delays anti-bloqueio

In [ ]:
# Executa o download
print("\nIniciando download...\n")
result = downloader.download_from_url(URL)

print("\n" + "="*60)
print("DOWNLOAD CONCLUIDO")
print("="*60)

### 3.3 - Exibir Estatisticas

Mostra o resumo do processamento e estatisticas do CSV.

In [ ]:
# Estatisticas do download atual
if result['success']:
    stats = result.get('stats', downloader.get_stats())
    
    print("ESTATISTICAS DO DOWNLOAD:")
    print("-"*60)
    print(f"Total processado:  {stats['total_attempted']}")
    print(f"Sucesso:           {stats['successful']}")
    print(f"Pulados:           {stats['skipped']}")
    print(f"Falhas:            {stats['failed']}")
    
    # Estatisticas do CSV consolidado
    print("\n" + "="*60)
    print("RESUMO DO CSV DE METADADOS")
    print("="*60)
    
    summary = metadata_manager.get_summary()
    print(f"Total de videos no banco:  {summary['total_videos']}")
    print(f"Tamanho total acumulado:   {summary['total_size_formatted']}")
    print(f"Duracao total acumulada:   {summary['total_duration_formatted']}")
    
    if summary['by_type']:
        print("\nPOR TIPO DE DOWNLOAD:")
        print("-"*60)
        for dtype, stats_type in summary['by_type'].items():
            if stats_type['count'] > 0:
                print(f"{dtype.upper():12} {stats_type['count']:4} videos  |  {stats_type['size_formatted']:10}  |  {stats_type['duration_formatted']}")
    
    print("="*60)
else:
    print(f"ERRO: {result.get('error', 'Desconhecido')}")

## SECAO 4: Validacao e Analise

Nesta secao:
1. Verificar CSV gerado
2. Visualizar dados com pandas
3. Analises estatisticas
4. Listar arquivos baixados

### 4.1 - Verificar CSV de Metadados

Carrega e exibe o conteudo do CSV consolidado.

In [ ]:
import pandas as pd
from downloaders import MetadataManager

# Carrega o CSV
manager = MetadataManager()
df = manager.load_csv()

print("="*60)
print(f"CSV DE METADADOS ({len(df)} registros)")
print("="*60)

if not df.empty:
    # Exibe colunas disponiveis
    print(f"\nColunas: {len(df.columns)}")
    print("-"*60)
    for i, col in enumerate(df.columns, 1):
        print(f"{i:2}. {col}")
    
    # Preview dos dados (primeiras 10 linhas, colunas principais)
    print("\n" + "="*60)
    print("PREVIEW DOS DADOS (primeiras linhas)")
    print("="*60)
    
    display_columns = ['id', 'title', 'uploader', 'duration', 'view_count', 'audio_format', 'download_date']
    available_columns = [col for col in display_columns if col in df.columns]
    
    # Configura pandas para exibir mais informacoes
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 50)
    pd.set_option('display.width', None)
    
    print(df[available_columns].head(10).to_string(index=False))
    
    print("\n" + "="*60)
else:
    print("\nCSV vazio - nenhum download realizado ainda")
    print("="*60)

### 4.2 - Analises Estatisticas

Analisa os dados do CSV com pandas.

In [ ]:
import pandas as pd
from downloaders import MetadataManager

manager = MetadataManager()
df = manager.load_csv()

if not df.empty:
    print("="*60)
    print("ANALISES ESTATISTICAS")
    print("="*60)
    
    # Estatisticas basicas
    print("\n1. ESTATISTICAS GERAIS:")
    print("-"*60)
    print(f"Total de videos:        {len(df)}")
    print(f"Total de views:         {df['view_count'].sum():,.0f}" if 'view_count' in df.columns else "Views: N/A")
    print(f"Total de likes:         {df['like_count'].sum():,.0f}" if 'like_count' in df.columns else "Likes: N/A")
    print(f"Duracao total:          {df['duration'].sum() / 3600:.1f} horas" if 'duration' in df.columns else "Duracao: N/A")
    print(f"Duracao media:          {df['duration'].mean() / 60:.1f} minutos" if 'duration' in df.columns else "Duracao media: N/A")
    
    # Top 5 videos por views
    if 'view_count' in df.columns and 'title' in df.columns:
        print("\n2. TOP 5 VIDEOS MAIS VISTOS:")
        print("-"*60)
        top_views = df.nlargest(5, 'view_count')[['title', 'view_count', 'uploader']]
        for idx, row in top_views.iterrows():
            title = row['title'][:45] + "..." if len(row['title']) > 45 else row['title']
            print(f"{title}")
            print(f"  {row['view_count']:,.0f} views | {row['uploader']}")
            print()
    
    # Videos por canal
    if 'uploader' in df.columns:
        print("\n3. VIDEOS POR CANAL:")
        print("-"*60)
        by_channel = df.groupby('uploader').size().sort_values(ascending=False).head(10)
        for channel, count in by_channel.items():
            print(f"{channel:40} {count:3} videos")
    
    # Distribuicao por formato
    if 'audio_format' in df.columns:
        print("\n4. DISTRIBUICAO POR FORMATO:")
        print("-"*60)
        by_format = df.groupby('audio_format').size()
        for fmt, count in by_format.items():
            print(f"{fmt.upper():10} {count:4} arquivos")
    
    print("\n" + "="*60)
else:
    print("Nenhum dado disponivel para analise")

### 4.3 - Listar Arquivos Baixados

Lista todos os arquivos de audio baixados com seus tamanhos.

In [ ]:
from pathlib import Path
from config import Config
from utils import format_size

base_path = Config.get_base_path()

if base_path.exists():
    print("="*60)
    print(f"ARQUIVOS EM: {base_path}")
    print("="*60)
    
    # Busca todos os arquivos de audio
    audio_extensions = ['*.mp3', '*.flac', '*.wav', '*.m4a', '*.opus', '*.ogg']
    audio_files = []
    
    for ext in audio_extensions:
        audio_files.extend(base_path.rglob(ext))
    
    if audio_files:
        # Ordena por tamanho (maior primeiro)
        audio_files.sort(key=lambda f: f.stat().st_size, reverse=True)
        
        total_size = sum(f.stat().st_size for f in audio_files)
        
        print(f"\nTotal de arquivos: {len(audio_files)}")
        print(f"Tamanho total:     {format_size(total_size)}")
        print("\n" + "-"*60)
        print(f"{'ARQUIVO':<40} {'TAMANHO':>15}")
        print("-"*60)
        
        for audio_file in audio_files[:20]:  # Mostra primeiros 20
            size = audio_file.stat().st_size
            rel_path = audio_file.relative_to(base_path)
            filename = str(rel_path)[:40]
            print(f"{filename:<40} {format_size(size):>15}")
        
        if len(audio_files) > 20:
            print(f"\n... e mais {len(audio_files) - 20} arquivos")
        
        print("="*60)
    else:
        print("\nNenhum arquivo de audio encontrado")
        print("="*60)
else:
    print(f"Pasta base nao existe: {base_path}")

### 4.4 - Exportar Dados Filtrados (Opcional)

Exporta subconjuntos do CSV baseado em filtros.

In [ ]:
from downloaders import MetadataManager

manager = MetadataManager()

# Exemplo 1: Exportar apenas playlists
# manager.export_filtered({'download_type': 'playlist'}, 'playlists_only.csv')

# Exemplo 2: Exportar apenas videos de um canal especifico
# manager.export_filtered({'uploader': 'Nome do Canal'}, 'canal_especifico.csv')

# Exemplo 3: Exportar apenas MP3
# manager.export_filtered({'audio_format': 'mp3'}, 'apenas_mp3.csv')

print("="*60)
print("EXPORTACAO FILTRADA")
print("="*60)
print("\nPara exportar dados filtrados, descomente uma das linhas acima.")
print("\nExemplos de filtros:")
print("  - Por tipo:    {'download_type': 'playlist'}")
print("  - Por canal:   {'uploader': 'Nome do Canal'}")
print("  - Por formato: {'audio_format': 'mp3'}")
print("="*60)

## SECAO 5: Troubleshooting

Ferramentas para diagnostico e solucao de problemas.

### 5.1 - Testar Conectividade do yt-dlp

Verifica se o yt-dlp consegue acessar o YouTube.

In [ ]:
# Testa com um video curto conhecido
!yt-dlp --dump-json --no-warnings "https://www.youtube.com/watch?v=dQw4w9WgXcQ" | head -20

print("\n" + "="*60)
print("Se voce viu JSON acima, o yt-dlp esta funcionando!")
print("="*60)

### 5.2 - Verificar Logs de Erro

Exibe os ultimos erros registrados.

In [ ]:
from config import Config
from pathlib import Path

log_path = Config.get_log_path()

if log_path.exists():
    error_logs = list(log_path.glob('errors_*.log'))
    
    if error_logs:
        # Pega o log mais recente
        latest_error_log = max(error_logs, key=lambda p: p.stat().st_mtime)
        
        print("="*60)
        print(f"LOG DE ERROS: {latest_error_log.name}")
        print("="*60)
        
        with open(latest_error_log, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            # Mostra ultimas 20 linhas
            for line in lines[-20:]:
                print(line.rstrip())
        
        print("\n" + "="*60)
    else:
        print("Nenhum log de erro encontrado (bom sinal!)")
else:
    print("Pasta de logs nao existe ainda")

### 5.3 - Atualizar yt-dlp

Atualiza o yt-dlp para a versao mais recente (resolve muitos problemas).

In [ ]:
print("Atualizando yt-dlp...\n")
!pip install --upgrade yt-dlp

print("\n" + "="*60)
print("Versao do yt-dlp:")
!yt-dlp --version
print("="*60)

### 5.4 - Limpar Cache do yt-dlp

Remove cache que pode estar causando problemas.

In [ ]:
!rm -rf ~/.cache/yt-dlp/

print("="*60)
print("Cache do yt-dlp limpo com sucesso!")
print("="*60)

### 5.5 - Verificar Integridade dos Modulos

Testa se todos os modulos foram importados corretamente.

In [ ]:
print("="*60)
print("VERIFICACAO DE MODULOS")
print("="*60)

modules_ok = True

try:
    from config import Config
    print("[OK] Config")
except Exception as e:
    print(f"[ERRO] Config: {e}")
    modules_ok = False

try:
    from downloaders import YouTubeDownloader
    print("[OK] YouTubeDownloader")
except Exception as e:
    print(f"[ERRO] YouTubeDownloader: {e}")
    modules_ok = False

try:
    from downloaders import MetadataManager
    print("[OK] MetadataManager")
except Exception as e:
    print(f"[ERRO] MetadataManager: {e}")
    modules_ok = False

try:
    from downloaders import DriveManager
    print("[OK] DriveManager")
except Exception as e:
    print(f"[ERRO] DriveManager: {e}")
    modules_ok = False

try:
    import pandas as pd
    print(f"[OK] pandas (versao {pd.__version__})")
except Exception as e:
    print(f"[ERRO] pandas: {e}")
    modules_ok = False

print("="*60)
if modules_ok:
    print("Todos os modulos estao funcionando corretamente!")
else:
    print("ATENCAO: Alguns modulos tem problemas. Veja os erros acima.")
print("="*60)

---

## INFORMACOES ADICIONAIS

### Estrutura de Arquivos Gerada

```
/content/drive/MyDrive/Katube_Download/
├── metadata_complete.csv          # CSV consolidado com todos os metadados
├── database.json                   # Database JSON (compatibilidade)
├── video_{id}/                     # Videos individuais
│   └── {id}.mp3
├── Playlist_{id}/                  # Playlists
│   ├── {video_id1}/
│   │   └── {video_id1}.mp3
│   └── {video_id2}/
│       └── {video_id2}.mp3
├── Canal_{id}/                     # Canais
├── txt_{id}/                       # Downloads de arquivos txt
└── logs/                           # Logs de downloads e erros
    ├── download_YYYY-MM-DD.log
    └── errors_YYYY-MM-DD.log
```

### Campos do CSV de Metadados (23 colunas)

**Identificacao:**
- `id` - ID unico do video (chave primaria)

**Informacoes Basicas:**
- `title` - Titulo do video
- `description` - Descricao completa
- `duration` - Duracao em segundos
- `upload_date` - Data de upload (YYYYMMDD)
- `timestamp` - Timestamp Unix do upload

**Canal/Uploader:**
- `uploader` - Nome do canal
- `uploader_id` - ID do uploader
- `uploader_url` - URL do uploader
- `channel` - Nome do canal oficial
- `channel_id` - ID do canal
- `channel_url` - URL do canal

**Estatisticas:**
- `view_count` - Numero de visualizacoes
- `like_count` - Numero de likes
- `comment_count` - Numero de comentarios
- `average_rating` - Avaliacao media

**Categorizacao:**
- `categories` - Categorias do video
- `tags` - Tags/palavras-chave
- `language` - Idioma do video

**Arquivo:**
- `audio_format` - Formato do audio (mp3, flac, etc)
- `file_path` - Caminho completo do arquivo
- `file_size_bytes` - Tamanho em bytes
- `download_date` - Data e hora do download
- `download_type` - Tipo (video, playlist, channel, txt)

### Sistema de Skip Inteligente

O sistema verifica **3 condicoes** antes de baixar cada video:

1. **Arquivo existe?** - Verifica se o arquivo de audio ja foi baixado
2. **ID no CSV?** - Verifica se o ID esta registrado no metadata_complete.csv
3. **Pasta existe?** - Verifica se a pasta do video existe e nao esta vazia

Se **qualquer uma** dessas condicoes for verdadeira, o download e **pulado automaticamente** com uma mensagem indicando o motivo.

### Formatos de Audio Suportados

| Formato | Descricao | Uso Recomendado |
|---------|-----------|----------------|
| **mp3** | Universal, compativel | Musica em geral, compatibilidade maxima |
| **flac** | Sem perdas, alta qualidade | Audiophiles, arquivamento, musica classica |
| **wav** | Sem perdas, sem compressao | Edicao profissional, maximo detalhamento |
| **m4a** | Boa compressao | Dispositivos Apple, podcasts |
| **opus** | Melhor compressao moderna | Podcasts, economizar espaco |
| **ogg** | Open-source | Alternativa livre ao MP3 |

### Qualidade de Audio Recomendada

| Formato | Qualidade Recomendada | Motivo |
|---------|----------------------|--------|
| FLAC/WAV | 0 (best) | Formatos sem perdas, captura maxima qualidade |
| MP3 | 256-320 kbps | Equilibrio entre qualidade e tamanho |
| M4A | 256 kbps | Boa compressao, qualidade alta |
| OPUS | 128 kbps | Codec moderno, 128 kbps = qualidade excelente |
| OGG | 192 kbps | Boa qualidade para formato livre |

### Links de Suporte

- **Repositorio:** https://github.com/DosAnjos-AI/katube-colab
- **Issues:** https://github.com/DosAnjos-AI/katube-colab/issues
- **Documentacao yt-dlp:** https://github.com/yt-dlp/yt-dlp
- **Pandas Docs:** https://pandas.pydata.org/docs/

### Notas de Uso

1. **Respeite os termos de servico do YouTube**
2. **Use delays apropriados** entre downloads para evitar bloqueios
3. **Monitore o espaco do Drive** - arquivos de audio ocupam espaco
4. **Mantenha o yt-dlp atualizado** - problemas de download sao geralmente resolvidos com atualizacao
5. **O CSV e incremental** - novos downloads sao adicionados, nunca sobrescritos
6. **Backup do CSV** - considere fazer backup periodico do metadata_complete.csv

### Solucao de Problemas Comuns

**Erro de autenticacao/bot detection:**
- Aumente os delays (DELAY_MIN e DELAY_MAX)
- Aguarde alguns minutos antes de tentar novamente
- Atualize o yt-dlp (Secao 5.3)

**Download muito lento:**
- Normal para playlists grandes
- Os delays anti-bloqueio sao necessarios

**CSV vazio apos downloads:**
- Verifique se houve erros no download
- Apenas downloads bem-sucedidos sao salvos no CSV

**Arquivo nao foi criado:**
- Verifique espaco no Drive
- Verifique logs de erro (Secao 5.2)
- Video pode estar indisponivel ou privado

---

**Versao:** 2.0.0 | **Licenca:** MIT | **Criado com:** Google Colab + yt-dlp + pandas